# Single-Sample Case Study — Diagnosis, SHAP & DiCE

Walks through the full online pipeline for one specific test sample (Fault 13, sample 400): global detection, the 5-criteria TOPSIS ranking among competing specialists, then SHAP local explainability and DiCE counterfactual recommendations for the winning diagnosis. This is the script behind the case study in the thesis (section 4.6).

In [ ]:
import os
import joblib
import warnings
import pyreadr
import numpy as np
import pandas as pd

# Silencia avisos técnicos
warnings.filterwarnings('ignore')

# ==============================================================================
# NOTE: paths below are relative to this notebook living in /notebooks.
# Place the TEP dataset files in /data and trained model artifacts in /models
# (see the repository README for download links and folder layout).
# 1. CONFIGURAÇÃO E SIMULAÇÃO DE ENTRADA DE DADOS (VIA TESTING)
# ==============================================================================
PASTA_MODELOS = os.path.join("..", "models", "baseline")
# --- APONTE AQUI PARA O SEU ARQUIVO DE TESTE REAL ---
CAMINHO_TESTE = os.path.join("..", "data", "TEP_Faulty_Testing.RData")

print("📡 Conectando ao Banco de Dados da Planta (Ambiente de Teste)...")
res_r = pyreadr.read_r(CAMINHO_TESTE)
df_teste_completo = res_r[list(res_r.keys())[0]]

# Qual falha queremos simular?
FALHA_ALVO = 13
df_falha_alvo = df_teste_completo[df_teste_completo['faultNumber'] == FALHA_ALVO].reset_index(drop=True)

# ---> TESTE DO ONE-VS-REST <---
# Pegamos a linha 400 da simulação de teste (onde o caos já começou)
LINHA_TESTE = 400 
print(f"⏱️ Capturando amostra do instante t={LINHA_TESTE} da Falha {FALHA_ALVO}...")

# Extrai apenas os sensores, removendo os metadados
amostra_viva = df_falha_alvo.iloc[[LINHA_TESTE]].drop(columns=['faultNumber', 'simulationRun', 'sample', 'target'], errors='ignore')
# ==============================================================================
# 2. FASE 1: O PORTEIRO (DETECTOR GLOBAL)
# ==============================================================================
print("\n🛡️ [FASE 1] Inspecionando com Detector Global...")
pacote_global = joblib.load(os.path.join(PASTA_MODELOS, "detector_global.joblib"))
modelo_global = pacote_global['modelo']
features_globais = pacote_global['features']

status_planta = modelo_global.predict(amostra_viva[features_globais])[0]

if status_planta == 0:
    print("🟢 STATUS: NORMAL. Planta operando em regime estável.")
else:
    print("🔴 STATUS: ANOMALIA DETECTADA!")
    print("🚨 [FASE 2] Acionando Comitê de Especialistas (Super-TOPSIS 5-Critérios)...")

    # ==========================================================================
    # 3. FASE 2: COMITÊ DE ESPECIALISTAS + TOPSIS MULTICRITÉRIO
    # ==========================================================================
    comite = []
    FALHAS_EXCLUIDAS = [3, 9, 15] 

    for arquivo in os.listdir(PASTA_MODELOS):
        if arquivo.startswith("especialista_f") and arquivo.endswith(".joblib"):
            pacote = joblib.load(os.path.join(PASTA_MODELOS, arquivo))
            if pacote['falha_id'] not in FALHAS_EXCLUIDAS:
                comite.append(pacote)

    votos = []
    for esp in comite:
        dados_filtrados = amostra_viva[esp['features']]
        prob = esp['modelo'].predict_proba(dados_filtrados)[0][1]
        
        # Extração do Perfil Estatístico do Especialista
        votos.append({
            'Falha': f"Falha {esp['falha_id']}",
            'Probabilidade': prob,
            'Acc': esp['acuracia'],
            'Prec': esp.get('precisao', 0.95), # .get por segurança caso o arquivo seja antigo
            'Rec': esp.get('recall', 0.95),
            'F1': esp.get('f1', 0.95)
        })

    df_votos = pd.DataFrame(votos)

    # --- DECISÃO MULTICRITÉRIO (TOPSIS REFORMULADO) ---
    # Definição das colunas e pesos (Total 1.0)
    colunas_criterio = ['Probabilidade', 'Acc', 'Prec', 'Rec', 'F1']
    pesos = np.array([0.30, 0.10, 0.15, 0.10, 0.20])
    
    matriz = df_votos[colunas_criterio].values
    
    # 1. Normalização Vetorial
    norma = np.linalg.norm(matriz, axis=0) + 1e-10
    matriz_norm = (matriz / norma) * pesos
    
    # 2. Identificação dos Ideais (Maximizar todos os critérios)
    id_pos = np.max(matriz_norm, axis=0)
    id_neg = np.min(matriz_norm, axis=0)
    
    # 3. Cálculo das Distâncias Euclidianas
    dist_pos = np.sqrt(np.sum((matriz_norm - id_pos)**2, axis=1))
    dist_neg = np.sqrt(np.sum((matriz_norm - id_neg)**2, axis=1))
    
    # 4. Score Final de Proximidade (Confiança Relativa)
    df_votos['Score_TOPSIS'] = dist_neg / (dist_pos + dist_neg + 1e-10)
    
    df_ranking = df_votos.sort_values(by='Score_TOPSIS', ascending=False).reset_index(drop=True)
    vencedor = df_ranking.iloc[0]

    # ==========================================================================
    # 4. VEREDICTO FINAL E PAINEL DE TRANSPARÊNCIA (XAI)
    # ==========================================================================
    print("\n" + "="*80)
    print(f"🎯 DIAGNÓSTICO FINAL CRAVADO: {vencedor['Falha']}")
    print("="*80)
    print(f"🔹 O que os sensores dizem AGORA (Probabilidade) : {vencedor['Probabilidade']*100:.2f}%")
    print(f"🔹 Nível de Confiança Multicritério (TOPSIS)     : {vencedor['Score_TOPSIS']:.4f}")
    print("-"*80)
    
    print("\n📊 MATRIZ DE AVALIAÇÃO MULTICRITÉRIO (TOP 3 SUSPEITOS):")
    
    # Prepara o DataFrame para ficar com uma formatação linda no terminal
    df_painel = df_ranking.head(3).copy()
    df_painel['Prob Atual'] = (df_painel['Probabilidade'] * 100).apply(lambda x: f"{x:.2f}%")
    df_painel['Acurácia'] = (df_painel['Acc'] * 100).apply(lambda x: f"{x:.1f}%")
    df_painel['Precisão'] = (df_painel['Prec'] * 100).apply(lambda x: f"{x:.1f}%")
    df_painel['Recall'] = (df_painel['Rec'] * 100).apply(lambda x: f"{x:.1f}%")
    df_painel['F1-Score'] = (df_painel['F1'] * 100).apply(lambda x: f"{x:.1f}%")
    df_painel['Score TOPSIS'] = df_painel['Score_TOPSIS'].apply(lambda x: f"{x:.4f}")

    # Escolhe a ordem das colunas para exibir
    colunas_exibir = ['Falha', 'Score TOPSIS', 'Prob Atual', 'Acurácia', 'Precisão', 'Recall', 'F1-Score']
    print(df_painel[colunas_exibir].to_string(index=False))

    print("\n✨ PRÓXIMAS ETAPAS PRESCRITIVAS:")
    print(f"   1. SHAP Local : Identificar os gargalos (NCA) da {vencedor['Falha']} hoje.")
    print(f"   2. DiCE       : Gerar rota contrafatual de resgate.")
    print("="*80)

In [ ]:
import shap
import dice_ml
import matplotlib.pyplot as plt
from IPython.display import display

print("\n" + "="*60)
print("🚀 INICIANDO PROTOCOLO DE INTELIGÊNCIA EXPLICÁVEL (XAI)")
print("="*60)

# 1. Recuperando os dados do Especialista Vencedor
id_vencedor = int(vencedor['Falha'].split(" ")[1]) # Pega o número 13
pacote_vencedor = joblib.load(os.path.join(PASTA_MODELOS, f"especialista_f{id_vencedor}.joblib"))
modelo_vencedor = pacote_vencedor['modelo']
features_vencedor = pacote_vencedor['features']

# A amostra exata do momento do caos (Linha 400), filtrada apenas para os 10 sensores do especialista
amostra_caos = amostra_viva[features_vencedor]

# ==============================================================================
# ETAPA 1: SHAP LOCAL (O DIAGNÓSTICO DO MOMENTO)
# ==============================================================================
print(f"\n🔍 [ETAPA 1] Executando SHAP Local para a {vencedor['Falha']}...")

explainer_local = shap.TreeExplainer(modelo_vencedor)
shap_values_local = explainer_local(amostra_caos)

# Mostra no terminal quais variáveis mais puxaram a probabilidade para 100%
df_shap = pd.DataFrame({
    'Sensor': features_vencedor,
    'Valor_Atual': amostra_caos.values[0],
    'Impacto_SHAP': shap_values_local.values[0]
})
df_shap = df_shap.sort_values(by='Impacto_SHAP', ascending=False)

print("\n🚨 SENSORES CULPADOS (Maior impacto para confirmar a falha):")
print(df_shap.head(3).to_string(index=False))

# Gera o gráfico Waterfall (Lindo para colocar no PDF do TCC!)
print("\n📊 Gerando gráfico Waterfall do SHAP...")
shap.plots.waterfall(shap_values_local[0], show=False)
plt.title(f"Justificativa do Modelo para a {vencedor['Falha']} (Amostra 400)")
plt.tight_layout()
plt.show()

# ==============================================================================
# ETAPA 2: DiCE (A PRESCRIÇÃO / RECEITA MÉDICA)
# ==============================================================================
print("\n💊 [ETAPA 2] Acionando DiCE para gerar Contrafatuais de Resgate...")
CAMINHO_NORMAL = os.path.join("..", "data", "TEP_FaultFree_Training.RData")
# O DiCE precisa conhecer os limites da planta saudável para saber para onde voltar.
# Vamos carregar um pouco do dataset Normal para servir de mapa para ele.
res_normal = pyreadr.read_r(CAMINHO_NORMAL)
df_mapa_normal = res_normal[list(res_normal.keys())[0]].sample(n=1000, random_state=42)
df_mapa_normal = df_mapa_normal[features_vencedor].copy()
df_mapa_normal['target'] = 0 # 0 = Estado Saudável

# A. Configurando os dados para o DiCE
d = dice_ml.Data(
    dataframe=df_mapa_normal, 
    continuous_features=features_vencedor, 
    outcome_name='target'
)

# B. Configurando o modelo para o DiCE entender
m = dice_ml.Model(model=modelo_vencedor, backend="sklearn")

# C. O Explicador DiCE (Usando Random, que é rápido e ótimo para o TEP)
exp_dice = dice_ml.Dice(d, m, method="random")

# D. Gerando a Receita: "O que eu mudo na amostra do CAOS para ela voltar a ser 0 (Normal)?"
print(f"⚙️ Calculando 3 rotas de resgate para estabilizar a planta...\n")
cf = exp_dice.generate_counterfactuals(
    amostra_caos, 
    total_CFs=3, 
    desired_class=0, # Queremos que a probabilidade de falha zere
    random_seed=42
)

# Mostra o resultado apenas com o que o operador precisa alterar
cf.visualize_as_dataframe(show_only_changes=True)

print("\n✅ CICLO DE DIAGNÓSTICO E PRESCRIÇÃO CONCLUÍDO COM SUCESSO!")